# RAG Pipeline — Phase 1

1. **TF-IDF Keyword Extraction** from train/test prompts
2. **Wikipedia Dump Loading** from HuggingFace (filtered by extracted keywords)
3. **Text Chunking** (~200 token passages with overlap)
4. **Embedding** with `all-MiniLM-L6-v2`
5. **FAISS Index** creation and saving to disk

**Environment**: Kaggle T4×2 GPU (32 GB VRAM)  
**Output**: `wiki_faiss.index` + `chunks.pkl` saved to `/kaggle/working/`

In [1]:
!pip install faiss-gpu --quiet

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 134.9/134.9 MB 13.3 MB/s eta 0:00:00


In [2]:
import pandas as pd
import numpy as np
import re
import pickle
import time
import os
from collections import Counter

from sklearn.feature_extraction.text import TfidfVectorizer
from datasets import load_dataset
from sentence_transformers import SentenceTransformer
from transformers import AutoTokenizer
import faiss

In [3]:
# For Kaggle:
TRAIN_CSV = "/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv"
TEST_CSV  = "/kaggle/input/competitions/smart-mcq-solver-challenge/test.csv"
OUTPUT_DIR = "/kaggle/working"

# For local:
# TRAIN_CSV = "../dataset/train.csv"
# TEST_CSV  = "../dataset/test.csv"
# OUTPUT_DIR = "../rag_artifacts"

os.makedirs(OUTPUT_DIR, exist_ok=True)

CHUNK_SIZE = 200 
CHUNK_OVERLAP = 40 

EMBEDDING_MODEL = "sentence-transformers/all-MiniLM-L6-v2"
EMBEDDING_DIM = 384

TOP_N_KEYWORDS = 150 
MAX_ARTICLES = 100_000  
MIN_ARTICLE_LEN = 500 

print(f"Output directory: {OUTPUT_DIR}")
print(f"Chunk size: {CHUNK_SIZE} tokens, overlap: {CHUNK_OVERLAP} tokens")

Output directory: /kaggle/working
Chunk size: 200 tokens, overlap: 40 tokens


In [4]:
train_df = pd.read_csv(TRAIN_CSV)
test_df  = pd.read_csv(TEST_CSV)

print(f"Train: {len(train_df)} rows, Test: {len(test_df)} rows")
print(f"Train columns: {list(train_df.columns)}")
print(f"\nSample prompt: {train_df['prompt'].iloc[0][:120]}...")

Train: 2000 rows, Test: 500 rows
Train columns: ['id', 'prompt', 'A', 'B', 'C', 'D', 'E', 'answer']

Sample prompt: Pick the best possible answer: What is Martin Heidegger's view on the relationship between time and human existence? amo...


In [5]:
def clean_prompt(text):
    preamble_patterns = [
        r"^Pick the best possible answer:\s*",
        r"^Select the most accurate option:\s*",
        r"^Determine the correct option:\s*",
        r"^Identify the correct statement:\s*",
        r"^Which of the following is correct\?\s*",
    ]
    for pattern in preamble_patterns:
        text = re.sub(pattern, "", text, flags=re.IGNORECASE)
    
    suffix_patterns = [
        r"\s*among the listed options\.?$",
        r"\s*from the following choices\.?$",
        r"\s*carefully\.?$",
    ]
    for pattern in suffix_patterns:
        text = re.sub(pattern, "", text, flags=re.IGNORECASE)
    
    return text.strip()

train_prompts_clean = train_df["prompt"].apply(clean_prompt).tolist()
test_prompts_clean  = test_df["prompt"].apply(clean_prompt).tolist()
all_prompts_clean   = train_prompts_clean + test_prompts_clean

print(f"Total prompts for TF-IDF: {len(all_prompts_clean)}")
print(f"\nOriginal:  {train_df['prompt'].iloc[0][:100]}")
print(f"Cleaned:   {train_prompts_clean[0][:100]}")

Total prompts for TF-IDF: 2500

Original:  Pick the best possible answer: What is Martin Heidegger's view on the relationship between time and 
Cleaned:   What is Martin Heidegger's view on the relationship between time and human existence?


In [6]:
tfidf = TfidfVectorizer(max_features=5000,stop_words="english",
                        ngram_range=(1, 2), min_df=2,max_df=0.8, token_pattern=r"(?u)\b[a-zA-Z][a-zA-Z]+\b")

tfidf_matrix = tfidf.fit_transform(all_prompts_clean)
feature_names = tfidf.get_feature_names_out()

print(f"TF-IDF vocabulary size: {len(feature_names)}")
print(f"TF-IDF matrix shape:    {tfidf_matrix.shape}")

TF-IDF vocabulary size: 1922
TF-IDF matrix shape:    (2500, 1922)


In [7]:
tfidf_scores = np.asarray(tfidf_matrix.sum(axis=0)).flatten()
top_indices  = tfidf_scores.argsort()[::-1][:TOP_N_KEYWORDS]

keywords_with_scores = [
    (feature_names[i], tfidf_scores[i]) for i in top_indices]

keywords = [kw for kw, score in keywords_with_scores]

print(f"\nTop {TOP_N_KEYWORDS} TF-IDF keywords extracted:")
print("="*60)
for i, (kw, score) in enumerate(keywords_with_scores[:30]):
    print(f"  {i+1:3d}. {kw:<35s} (score: {score:.2f})")
print(f"  ... and {len(keywords)-30} more")


Top 150 TF-IDF keywords extracted:
    1. based                               (score: 64.40)
    2. context                             (score: 61.94)
    3. based given                         (score: 61.27)
    4. given                               (score: 61.27)
    5. given context                       (score: 61.27)
    6. choose correct                      (score: 51.54)
    7. correct                             (score: 51.54)
    8. choose                              (score: 51.54)
    9. answer                              (score: 51.54)
   10. correct answer                      (score: 51.54)
   11. relationship                        (score: 32.02)
   12. effect                              (score: 31.65)
   13. definition                          (score: 23.71)
   14. used                                (score: 22.70)
   15. reason                              (score: 22.06)
   16. significance                        (score: 21.94)
   17. physics                      

In [8]:
keyword_set = set()
for kw in keywords:
    keyword_set.add(kw.lower())
    for word in kw.lower().split():
        if len(word) > 3:
            keyword_set.add(word)

print(f"Total unique keyword terms (including split bigrams): {len(keyword_set)}")
print(f"\nSample keywords: {sorted(list(keyword_set))[:20]}")

Total unique keyword terms (including split bigrams): 151

Sample keywords: ['acceleration', 'according', 'accurately', 'accurately describes', 'answer', 'astronomy', 'atomristor', 'balance', 'based', 'based given', 'black', 'book', 'butterfly', 'butterfly effect', 'choose', 'choose correct', 'concept', 'condition', 'context', 'convection']


In [9]:
def title_matches_keywords(title, keyword_set):
    title_lower = title.lower()
    title_words = set(re.findall(r'\b[a-z]+\b', title_lower))
    
    for kw in keyword_set:
        if ' ' not in kw:
            if kw in title_words:
                return True
        else:
            if kw in title_lower:
                return True
    return False

test_titles = ["Quantum mechanics",
    "History of pizza", "Redshift", "Martin Heidegger",
    "List of football clubs","Superconductivity"]

for t in test_titles:
    match = title_matches_keywords(t, keyword_set)
    print(f"{'✓' if match else '✗'} {t}")

✓ Quantum mechanics
✗ History of pizza
✓ Redshift
✗ Martin Heidegger
✗ List of football clubs
✗ Superconductivity


In [10]:
print("Loading Wikipedia dataset (streaming mode)...")
wiki_dataset = load_dataset("wikimedia/wikipedia","20231101.en",split="train",streaming=True)

filtered_articles = []
total_scanned = 0
start_time = time.time()

print("Scanning and filtering articles...")
print("(This will take a while — streaming through ~6.8M articles)\n")

for article in wiki_dataset:
    total_scanned += 1
    
    if total_scanned % 100_000 == 0:
        elapsed = time.time() - start_time
        rate = total_scanned / elapsed
        print(
            f"  Scanned: {total_scanned:>8,d} | "
            f"Matched: {len(filtered_articles):>6,d} | "
            f"Rate: {rate:,.0f} articles/sec | "
            f"Elapsed: {elapsed/60:.1f} min"
        )
    
    if len(article["text"]) < MIN_ARTICLE_LEN:
        continue
    
    if title_matches_keywords(article["title"], keyword_set):
        filtered_articles.append({
            "title": article["title"],
            "text":  article["text"],
            "url":   article.get("url", "")})
    
    if len(filtered_articles) >= MAX_ARTICLES:
        print(f"Reached MAX_ARTICLES cap ({MAX_ARTICLES}). Stopping scan.")
        break

elapsed = time.time() - start_time
print(f"Done! Scanned {total_scanned:,d} articles in {elapsed/60:.1f} minutes")
print(f"Matched {len(filtered_articles):,d} relevant articles")
print(f"Match rate: {len(filtered_articles)/total_scanned*100:.2f}%")

Loading Wikipedia dataset (streaming mode)...


README.md: 0.00B [00:00, ?B/s]

Resolving data files:   0%|          | 0/41 [00:00<?, ?it/s]

Scanning and filtering articles...
(This will take a while — streaming through ~6.8M articles)

  Scanned:  100,000 | Matched:  1,889 | Rate: 8,246 articles/sec | Elapsed: 0.2 min
  Scanned:  200,000 | Matched:  3,754 | Rate: 9,093 articles/sec | Elapsed: 0.4 min
  Scanned:  300,000 | Matched:  5,343 | Rate: 8,669 articles/sec | Elapsed: 0.6 min
  Scanned:  400,000 | Matched:  7,103 | Rate: 8,901 articles/sec | Elapsed: 0.7 min
  Scanned:  500,000 | Matched:  8,799 | Rate: 9,228 articles/sec | Elapsed: 0.9 min
  Scanned:  600,000 | Matched: 10,528 | Rate: 9,820 articles/sec | Elapsed: 1.0 min
  Scanned:  700,000 | Matched: 12,264 | Rate: 9,706 articles/sec | Elapsed: 1.2 min
  Scanned:  800,000 | Matched: 13,895 | Rate: 9,579 articles/sec | Elapsed: 1.4 min
  Scanned:  900,000 | Matched: 15,416 | Rate: 9,980 articles/sec | Elapsed: 1.5 min
  Scanned: 1,000,000 | Matched: 16,654 | Rate: 8,817 articles/sec | Elapsed: 1.9 min
  Scanned: 1,100,000 | Matched: 17,917 | Rate: 8,744 articles/s

'The read operation timed out' thrown while requesting GET https://huggingface.co/datasets/wikimedia/wikipedia/resolve/b04c8d1ceb2f5cd4588862100d08de323dccfbaa/20231101.en/train-00014-of-00041.parquet
Retrying in 1s [Retry 1/5].
'The read operation timed out' thrown while requesting GET https://huggingface.co/datasets/wikimedia/wikipedia/resolve/b04c8d1ceb2f5cd4588862100d08de323dccfbaa/20231101.en/train-00014-of-00041.parquet
Retrying in 1s [Retry 1/5].


...

  Scanned: 3,700,000 | Matched: 51,781 | Rate: 9,103 articles/sec | Elapsed: 6.8 min
  Scanned: 3,800,000 | Matched: 53,050 | Rate: 9,144 articles/sec | Elapsed: 6.9 min
  Scanned: 3,900,000 | Matched: 54,203 | Rate: 9,252 articles/sec | Elapsed: 7.0 min
  Scanned: 4,000,000 | Matched: 55,462 | Rate: 9,307 articles/sec | Elapsed: 7.2 min


'The read operation timed out' thrown while requesting GET https://huggingface.co/datasets/wikimedia/wikipedia/resolve/b04c8d1ceb2f5cd4588862100d08de323dccfbaa/20231101.en/train-00026-of-00041.parquet
Retrying in 1s [Retry 1/5].


  Scanned: 4,100,000 | Matched: 56,657 | Rate: 9,150 articles/sec | Elapsed: 7.5 min


'The read operation timed out' thrown while requesting GET https://huggingface.co/datasets/wikimedia/wikipedia/resolve/b04c8d1ceb2f5cd4588862100d08de323dccfbaa/20231101.en/train-00026-of-00041.parquet
Retrying in 1s [Retry 1/5].


  Scanned: 4,200,000 | Matched: 57,864 | Rate: 8,965 articles/sec | Elapsed: 7.8 min
  Scanned: 4,300,000 | Matched: 59,056 | Rate: 9,020 articles/sec | Elapsed: 7.9 min
  Scanned: 4,400,000 | Matched: 60,358 | Rate: 9,004 articles/sec | Elapsed: 8.1 min
  Scanned: 4,500,000 | Matched: 61,490 | Rate: 9,101 articles/sec | Elapsed: 8.2 min
  Scanned: 4,600,000 | Matched: 62,542 | Rate: 9,119 articles/sec | Elapsed: 8.4 min
  Scanned: 4,700,000 | Matched: 63,638 | Rate: 9,202 articles/sec | Elapsed: 8.5 min
  Scanned: 4,800,000 | Matched: 64,818 | Rate: 9,288 articles/sec | Elapsed: 8.6 min
  Scanned: 4,900,000 | Matched: 66,031 | Rate: 9,346 articles/sec | Elapsed: 8.7 min
  Scanned: 5,000,000 | Matched: 67,211 | Rate: 9,422 articles/sec | Elapsed: 8.8 min
  Scanned: 5,100,000 | Matched: 68,503 | Rate: 9,447 articles/sec | Elapsed: 9.0 min
  Scanned: 5,200,000 | Matched: 69,503 | Rate: 9,482 articles/sec | Elapsed: 9.1 min
  Scanned: 5,300,000 | Matched: 70,680 | Rate: 9,572 articles/sec

In [11]:
print(f"Total filtered articles: {len(filtered_articles)}")
print(f"\nArticle text length statistics:")

lengths = [len(a["text"]) for a in filtered_articles]
print(f"Mean:{np.mean(lengths):,.0f} chars")
print(f"Median:{np.median(lengths):,.0f} chars")
print(f"Min:{np.min(lengths):,d} chars")
print(f"Max:{np.max(lengths):,d} chars")

print(f"\nSample matched titles:")
for a in filtered_articles[:15]:
    print(f" {a['title']} ({len(a['text']):,d} chars)")

Total filtered articles: 89754

Article text length statistics:
Mean:5,192 chars
Median:2,522 chars
Min:500 chars
Max:325,241 chars

Sample matched titles:
 Animalia (book) (2,340 chars)
 International Atomic Time (7,487 chars)
 Answer (law) (1,649 chars)
 Albert Einstein (84,414 chars)
 Amateur astronomy (23,254 chars)
 Abstract (law) (1,272 chars)
 Algebraically closed field (8,267 chars)
 Aspect ratio (3,780 chars)
 Atomic physics (5,778 chars)
 Astronomical unit (21,555 chars)
 Adiabatic process (25,798 chars)
 The Triumph of Time (613 chars)
 Book of Amos (5,168 chars)
 Antimicrobial resistance (72,276 chars)
 Amdahl's law (9,984 chars)


In [12]:
articles_path = os.path.join(OUTPUT_DIR, "filtered_articles.pkl")
with open(articles_path, "wb") as f:
    pickle.dump(filtered_articles, f)

file_size_mb = os.path.getsize(articles_path) / (1024 * 1024)
print(f"Saved {len(filtered_articles):,d} articles to {articles_path}")
print(f"File size: {file_size_mb:.1f} MB")

Saved 89,754 articles to /kaggle/working/filtered_articles.pkl
File size: 455.6 MB


---
## Step 3A: Chunk Articles into ~200 Token Passages

Each Wikipedia article is split into overlapping passages of ~200 tokens.  
We use the **same tokenizer** as our embedding model to ensure accurate token counts.

**Parameters**:
- `CHUNK_SIZE = 200` tokens → ~150 words, a solid paragraph
- `CHUNK_OVERLAP = 40` tokens → 20% overlap to avoid losing facts at boundaries

In [13]:
chunk_tokenizer = AutoTokenizer.from_pretrained(EMBEDDING_MODEL)


def chunk_article(text, title, tokenizer, chunk_size=200, overlap=40):
    tokens = tokenizer.encode(text, add_special_tokens=False)
    
    if len(tokens) <= chunk_size:
        return [{
            "title": title,
            "text": tokenizer.decode(tokens, skip_special_tokens=True),
            "chunk_idx": 0}]
    
    chunks = []
    stride = chunk_size - overlap
    start = 0
    chunk_idx = 0
    
    while start < len(tokens):
        end = min(start + chunk_size, len(tokens))
        chunk_tokens = tokens[start:end]
        chunk_text = tokenizer.decode(chunk_tokens, skip_special_tokens=True)
        
        if len(chunk_tokens) >= 20:
            chunks.append({
                "title": title,
                "text": chunk_text,
                "chunk_idx": chunk_idx})
            chunk_idx += 1
        
        start += stride
    
    return chunks

sample = filtered_articles[0]
sample_chunks = chunk_article(sample["text"], sample["title"],
                              chunk_tokenizer,chunk_size=CHUNK_SIZE, overlap=CHUNK_OVERLAP)
print(f"Article: '{sample['title']}'")
print(f"Original length: {len(sample['text']):,d} chars")
print(f"Chunks produced: {len(sample_chunks)}")
print(f"First chunk preview ({len(sample_chunks[0]['text'])} chars):")
print(f"{sample_chunks[0]['text'][:200]}...")

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

Article: 'Animalia (book)'
Original length: 2,340 chars
Chunks produced: 3
First chunk preview (1008 chars):
animalia is an illustrated children ' s book by graeme base. it was originally published in 1986, followed by a tenth anniversary edition in 1996, and a 25th anniversary edition in 2012. over four mil...


In [14]:
all_chunks = []
start_time = time.time()

for i, article in enumerate(filtered_articles):
    chunks = chunk_article(article["text"], article["title"],chunk_tokenizer,
                           chunk_size=CHUNK_SIZE,overlap=CHUNK_OVERLAP)
    all_chunks.extend(chunks)
    
    if (i + 1) % 10_000 == 0:
        elapsed = time.time() - start_time
        print(
            f"Processed {i+1:>7,d}/{len(filtered_articles):,d} articles | "
            f"Total chunks: {len(all_chunks):>8,d} | "
            f"Elapsed: {elapsed:.1f}s"
        )
        

elapsed = time.time() - start_time
print(f"Chunking complete!")
print(f"Articles processed: {len(filtered_articles):,d}")
print(f"Total chunks: {len(all_chunks):,d}")
print(f"Avg chunks/article: {len(all_chunks)/len(filtered_articles):.1f}")
print(f"Time: {elapsed:.1f}s")

Token indices sequence length is longer than the specified maximum sequence length for this model (1559 > 512). Running this sequence through the model will result in indexing errors


Processed  10,000/89,754 articles | Total chunks:   75,587 | Elapsed: 56.6s
Processed  20,000/89,754 articles | Total chunks:  146,151 | Elapsed: 110.7s
Processed  30,000/89,754 articles | Total chunks:  209,979 | Elapsed: 161.2s
Processed  40,000/89,754 articles | Total chunks:  283,706 | Elapsed: 221.4s
Processed  50,000/89,754 articles | Total chunks:  346,589 | Elapsed: 273.1s
Processed  60,000/89,754 articles | Total chunks:  402,207 | Elapsed: 318.1s
Processed  70,000/89,754 articles | Total chunks:  461,575 | Elapsed: 366.4s
Processed  80,000/89,754 articles | Total chunks:  557,912 | Elapsed: 449.7s
Chunking complete!
Articles processed: 89,754
Total chunks: 646,914
Avg chunks/article: 7.2
Time: 524.9s


In [15]:
chunk_lengths = [len(c["text"]) for c in all_chunks]
chunk_token_lengths = [len(chunk_tokenizer.encode(c["text"], add_special_tokens=False)) for c in all_chunks[:1000]]

print("Chunk length statistics (characters):")
print(f"Mean: {np.mean(chunk_lengths):,.0f}")
print(f"Median: {np.median(chunk_lengths):,.0f}")
print(f"Min: {np.min(chunk_lengths):,d}")
print(f"Max: {np.max(chunk_lengths):,d}")

print(f"\nChunk length statistics (tokens, sampled from first 1000):")
print(f"Mean: {np.mean(chunk_token_lengths):.0f}")
print(f"Median: {np.median(chunk_token_lengths):.0f}")
print(f"Min: {np.min(chunk_token_lengths)}")
print(f"Max: {np.max(chunk_token_lengths)}")

Chunk length statistics (characters):
Mean: 891
Median: 943
Min: 30
Max: 1,715

Chunk length statistics (tokens, sampled from first 1000):
Mean: 196
Median: 200
Min: 21
Max: 203


In [16]:
chunks_path = os.path.join(OUTPUT_DIR, "chunks.pkl")
with open(chunks_path, "wb") as f:
    pickle.dump(all_chunks, f)

file_size_mb = os.path.getsize(chunks_path) / (1024 * 1024)
print(f"Saved {len(all_chunks):,d} chunks to {chunks_path}")
print(f"File size: {file_size_mb:.1f} MB")

Saved 646,914 chunks to /kaggle/working/chunks.pkl
File size: 567.1 MB


---
## Step 3B: Embed Chunks with `all-MiniLM-L6-v2`

We convert every chunk into a 384-dimensional dense vector.  
Embeddings are **L2-normalized** so that inner product = cosine similarity.

**Model**: `sentence-transformers/all-MiniLM-L6-v2` (22M parameters)  
**VRAM**: ~90 MB — runs comfortably even on CPU

In [17]:
embed_model = SentenceTransformer(EMBEDDING_MODEL)

import torch
device = "cuda" if torch.cuda.is_available() else "cpu"
embed_model = embed_model.to(device)
print(f"Embedding model loaded on: {device}")
print(f"Embedding dimension: {embed_model.get_sentence_embedding_dimension()}")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/90.9M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Embedding model loaded on: cuda
Embedding dimension: 384


/tmp/ipykernel_23/182640815.py:7: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  print(f"Embedding dimension: {embed_model.get_sentence_embedding_dimension()}")


In [18]:
from tqdm.asyncio import tqdm


chunk_texts_for_embedding = [
    f"{c['title']}: {c['text']}" for c in all_chunks
]

print(f"Embedding {len(chunk_texts_for_embedding):,d} chunks...")
print(f"Sample input: '{chunk_texts_for_embedding[0][:100]}...'\n")

start_time = time.time()

embeddings = embed_model.encode(chunk_texts_for_embedding, batch_size=128, 
                                show_progress_bar=True, normalize_embeddings=True, convert_to_numpy=True)

elapsed = time.time() - start_time

embeddings_path = os.path.join(OUTPUT_DIR, "embeddings.npy")
np.save(embeddings_path, embeddings)
emb_size_mb = os.path.getsize(embeddings_path) / (1024 * 1024)
print(f"Embeddings saved to:  {embeddings_path} ({emb_size_mb:.1f} MB)")

print(f"\nEmbedding complete!")
print(f"Shape: {embeddings.shape}")
print(f"Dtype:{embeddings.dtype}")
print(f"Time: {elapsed:.1f}s ({len(all_chunks)/elapsed:.0f} chunks/sec)")
print(f"Memory: {embeddings.nbytes / (1024**2):.1f} MB")

Embedding 646,914 chunks...
Sample input: 'Animalia (book): animalia is an illustrated children ' s book by graeme base. it was originally publ...'



Batches:   0%|          | 0/5055 [00:00<?, ?it/s]

Embeddings saved to:  /kaggle/working/embeddings.npy (947.6 MB)

Embedding complete!
Shape: (646914, 384)
Dtype:float32
Time: 1401.3s (462 chunks/sec)
Memory: 947.6 MB


In [19]:
test_query = "What is quantum entanglement?"
query_embedding = embed_model.encode([test_query], 
                                     normalize_embeddings=True,convert_to_numpy=True)
embeddings = np.load("/kaggle/working/embeddings.npy")
similarities = np.dot(embeddings, query_embedding.T).flatten()
top_5_indices = similarities.argsort()[::-1][:5]

print(f"Query: '{test_query}'")
print(f"\nTop 5 most similar chunks:")
for rank, idx in enumerate(top_5_indices):
    chunk = all_chunks[idx]
    print(f"\n  #{rank+1} (similarity: {similarities[idx]:.4f})")
    print(f"Title: {chunk['title']}")
    print(f"Text:  {chunk['text'][:150]}...")

Query: 'What is quantum entanglement?'

Top 5 most similar chunks:

  #1 (similarity: 0.6799)
Title: Quantum complex network
Text:  systems. a qubit is a quantum object that, when measured, can be found to be in one of only two states, and that is used to transmit information. phot...

  #2 (similarity: 0.6497)
Title: Quantum key distribution
Text:  and measure protocols in contrast to classical physics, the act of measurement is an integral part of quantum mechanics. in general, measuring an unkn...

  #3 (similarity: 0.6118)
Title: Aspect's experiment
Text:  was awarded part of the 2022 nobel prize in physics. scientific and historical context the experiment must be placed in its historical and scientific ...

  #4 (similarity: 0.5965)
Title: Quantum game theory
Text:  each of the two strategies available to the players. when a measurement is made on the electron, it collapses to one of the base states, thus conveyin...

  #5 (similarity: 0.5893)
Title: Quantum mind
Text:  even when 

---
## Step 3C: Build and Save the FAISS Index

We create a FAISS index using **Inner Product** similarity (which equals cosine similarity when vectors are L2-normalized).

- For **<1M chunks**: `IndexFlatIP` (exact brute-force search) — fast enough and gives perfect recall.
- For **>1M chunks**: `IndexIVFFlat` (approximate search with inverted file) — faster but slightly lower recall.

In [20]:
embeddings = np.load("/kaggle/working/embeddings.npy")
num_chunks = embeddings.shape[0]
dimension  = embeddings.shape[1]

print(f"Building FAISS index...")
print(f"Vectors:   {num_chunks:,d}")
print(f"Dimension: {dimension}")

if num_chunks < 1_000_000:
    index = faiss.IndexFlatIP(dimension)
    print(f"  Index type: IndexFlatIP (exact search)")
else:
    nlist = min(int(np.sqrt(num_chunks)), 4096)
    quantizer = faiss.IndexFlatIP(dimension)
    index = faiss.IndexIVFFlat(quantizer, dimension, nlist, faiss.METRIC_INNER_PRODUCT)
    
    print(f"Index type: IndexIVFFlat (approximate search)")
    print(f"Training index with {nlist} clusters...")
    index.train(embeddings)
    index.nprobe = 20 
    print(f"Training complete. nprobe={index.nprobe}")

index.add(embeddings)
print(f"Index built! Total vectors: {index.ntotal:,d}")

Building FAISS index...
Vectors:   646,914
Dimension: 384
  Index type: IndexFlatIP (exact search)
Index built! Total vectors: 646,914


In [21]:
scores, indices = index.search(query_embedding, 5)

print(f"Query: '{test_query}'")
print(f"\nFAISS Top 5 results:")
for rank, (score, idx) in enumerate(zip(scores[0], indices[0])):
    chunk = all_chunks[idx]
    print(f" #{rank+1} (score: {score:.4f}, index: {idx})")
    print(f"Title: {chunk['title']}")
    print(f"Text:  {chunk['text'][:150]}...")

Query: 'What is quantum entanglement?'

FAISS Top 5 results:
 #1 (score: 0.6799, index: 355199)
Title: Quantum complex network
Text:  systems. a qubit is a quantum object that, when measured, can be found to be in one of only two states, and that is used to transmit information. phot...
 #2 (score: 0.6497, index: 258110)
Title: Quantum key distribution
Text:  and measure protocols in contrast to classical physics, the act of measurement is an integral part of quantum mechanics. in general, measuring an unkn...
 #3 (score: 0.6118, index: 417229)
Title: Aspect's experiment
Text:  was awarded part of the 2022 nobel prize in physics. scientific and historical context the experiment must be placed in its historical and scientific ...
 #4 (score: 0.5965, index: 50888)
Title: Quantum game theory
Text:  each of the two strategies available to the players. when a measurement is made on the electron, it collapses to one of the base states, thus conveyin...
 #5 (score: 0.5893, index: 60629)
Title

In [22]:
import pickle, numpy as np, faiss

with open("/kaggle/working/chunks.pkl", "rb") as f:
    all_chunks = pickle.load(f)

embeddings = np.load("/kaggle/working/embeddings.npy")

index = faiss.IndexFlatIP(384)
index.add(embeddings)
del embeddings

from sentence_transformers import SentenceTransformer
embed_model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")
test_query = "What is quantum entanglement?"
query_embedding = embed_model.encode([test_query], normalize_embeddings=True, convert_to_numpy=True)
del embed_model

scores, indices = index.search(query_embedding, 5)
for rank, (score, idx) in enumerate(zip(scores[0], indices[0])):
    chunk = all_chunks[idx]
    print(f"#{rank+1} (score: {score:.4f}) {chunk['title']}: {chunk['text'][:150]}...")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


#1 (score: 0.6799) Quantum complex network: systems. a qubit is a quantum object that, when measured, can be found to be in one of only two states, and that is used to transmit information. phot...
#2 (score: 0.6497) Quantum key distribution: and measure protocols in contrast to classical physics, the act of measurement is an integral part of quantum mechanics. in general, measuring an unkn...
#3 (score: 0.6118) Aspect's experiment: was awarded part of the 2022 nobel prize in physics. scientific and historical context the experiment must be placed in its historical and scientific ...
#4 (score: 0.5965) Quantum game theory: each of the two strategies available to the players. when a measurement is made on the electron, it collapses to one of the base states, thus conveyin...
#5 (score: 0.5893) Quantum mind: even when the particles are separated by a large distance. instead, a quantum state has to be described for the whole system. measurements of physical...


In [23]:
# ─── Save Everything to Disk ─────────────────────────────────────

# 1. FAISS index
index_path = os.path.join(OUTPUT_DIR, "wiki_faiss.index")
faiss.write_index(index, index_path)
index_size_mb = os.path.getsize(index_path) / (1024 * 1024)
print(f"FAISS index saved to: {index_path} ({index_size_mb:.1f} MB)")

# 2. Raw embeddings (for potential re-indexing with different params)
# embeddings_path = os.path.join(OUTPUT_DIR, "embeddings.npy")
# np.save(embeddings_path, embeddings)
# emb_size_mb = os.path.getsize(embeddings_path) / (1024 * 1024)
# print(f"Embeddings saved to:  {embeddings_path} ({emb_size_mb:.1f} MB)")

# 3. Chunks already saved earlier (chunks.pkl)
print(f"Chunks saved to:      {chunks_path}")

FAISS index saved to: /kaggle/working/wiki_faiss.index (947.6 MB)
Chunks saved to:      /kaggle/working/chunks.pkl


In [24]:
print("RAG PHASE 1 — INDEXING COMPLETE")
print(f"TF-IDF keywords extracted: {len(keywords)}")
print(f"Wikipedia articles matched: {len(filtered_articles):,d}")
print(f"Total chunks created: {len(all_chunks):,d}")
print(f"Embedding dimension: {dimension}")
print(f"FAISS index vectors: {index.ntotal:,d}")

RAG PHASE 1 — INDEXING COMPLETE
TF-IDF keywords extracted: 150
Wikipedia articles matched: 89,754
Total chunks created: 646,914
Embedding dimension: 384
FAISS index vectors: 646,914


## RAG_Step4_to_8_Retrieval

<!-- merged from: RAG_Step4_to_8_Retrieval.ipynb (Jupy Tools) -->

# RAG Pipeline — Phase 2: Online Retrieval & Prompt Construction

This notebook covers **Steps 4–8** of the RAG pipeline:
- **Step 4**: Load the FAISS index and chunks from Phase 1
- **Step 5**: Embed question prompts using the same `all-MiniLM-L6-v2` model
- **Step 6**: FAISS search — retrieve top-k=10 passages per question
- **Step 7**: Cross-encoder re-ranking — narrow down to top 3 passages
- **Step 8**: Construct the augmented prompt (context + question + options)

**Prerequisites**: Run `RAG_Step1_to_3_Indexing.ipynb` first to generate:
- `wiki_faiss.index`
- `chunks.pkl`

**Environment**: Kaggle T4×2 GPU (32 GB VRAM)  
**Output**: Augmented prompts ready for LLM inference (Step 9)

In [25]:
import pandas as pd
import numpy as np
import pickle
import time
import os
import re

from sentence_transformers import SentenceTransformer, CrossEncoder
import faiss
import torch

print("All imports successful!")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

All imports successful!
CUDA available: True
GPU: Tesla T4


In [26]:
# ─── Configuration ───────────────────────────────────────────────
# Paths — adjust for Kaggle vs local

# For Kaggle:
TRAIN_CSV = "/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv"
TEST_CSV = "/kaggle/input/competitions/smart-mcq-solver-challenge/test.csv"
ARTIFACTS_DIR = "/kaggle/working"

# # For local:
# TRAIN_CSV     = "../dataset/train.csv"
# TEST_CSV      = "../dataset/test.csv"
# ARTIFACTS_DIR = "../rag_artifacts"           # must match Phase 1 OUTPUT_DIR

# Model names
EMBEDDING_MODEL = "sentence-transformers/all-MiniLM-L6-v2"
CROSS_ENCODER_MODEL = "cross-encoder/ms-marco-MiniLM-L-6-v2"

# Retrieval parameters
FAISS_TOP_K = 10
RERANK_TOP_N = 2 

CHOICES = ["A", "B", "C", "D", "E"]

print(f"Artifacts directory: {ARTIFACTS_DIR}")
print(f"FAISS top-k: {FAISS_TOP_K} → Cross-encoder top-n: {RERANK_TOP_N}")

Artifacts directory: /kaggle/working
FAISS top-k: 10 → Cross-encoder top-n: 2


---
## Step 4: Load Phase 1 Artifacts (FAISS Index + Chunks)

Load the pre-built FAISS index and chunk metadata from Phase 1.  
These were saved by `RAG_Step1_to_3_Indexing.ipynb`.

In [27]:
# ─── Load FAISS Index ─────────────────────────────────────────────
index_path = os.path.join(ARTIFACTS_DIR, "wiki_faiss.index")
index = faiss.read_index(index_path)

print(f"FAISS index loaded from: {index_path}")
print(f" Total vectors: {index.ntotal:,d}")
print(f" Dimension:     {index.d}")

FAISS index loaded from: /kaggle/working/wiki_faiss.index
 Total vectors: 646,914
 Dimension:     384


In [28]:
chunks_path = os.path.join(ARTIFACTS_DIR, "chunks.pkl")
with open(chunks_path, "rb") as f:
    all_chunks = pickle.load(f)

print(f"Chunks loaded from: {chunks_path}")
print(f"  Total chunks: {len(all_chunks):,d}")
print(f"\n  Sample chunk (index 0):")
print(f"    Title: {all_chunks[0]['title']}")
print(f"    Text:  {all_chunks[0]['text'][:150]}...")

# Verify chunk count matches FAISS index
assert len(all_chunks) == index.ntotal, (
    f"Mismatch! Chunks={len(all_chunks)}, FAISS vectors={index.ntotal}")
print(f"\n  ✓ Chunk count matches FAISS index ({index.ntotal:,d} vectors)")

Chunks loaded from: /kaggle/working/chunks.pkl
  Total chunks: 646,914

  Sample chunk (index 0):
    Title: Animalia (book)
    Text:  animalia is an illustrated children ' s book by graeme base. it was originally published in 1986, followed by a tenth anniversary edition in 1996, and...

  ✓ Chunk count matches FAISS index (646,914 vectors)


In [29]:
train_df = pd.read_csv(TRAIN_CSV)
test_df  = pd.read_csv(TEST_CSV)

for c in CHOICES:
    train_df[c] = train_df[c].fillna("").astype(str)
    test_df[c]  = test_df[c].fillna("").astype(str)
train_df["prompt"] = train_df["prompt"].fillna("").astype(str)
test_df["prompt"]  = test_df["prompt"].fillna("").astype(str)

print(f"Train: {len(train_df)} rows | Test: {len(test_df)} rows")
print(f"\nSample test prompt (row 0): {test_df.iloc[0]['prompt'][:100]}...")

Train: 2000 rows | Test: 500 rows

Sample test prompt (row 0): Pick the best possible answer: What is the relationship between the Hamiltonians and eigenstates in ...


---
## Step 5: Load Embedding Model & Embed Question Prompts

We use the **same** `all-MiniLM-L6-v2` model that was used to embed the Wikipedia chunks.  
This is critical — the query embedding must live in the same vector space as the chunk embeddings.

**What to embed**: Just the `prompt` field (the question itself). Including the options would dilute the semantic signal for retrieval.

In [30]:
embed_model = SentenceTransformer(EMBEDDING_MODEL)

device = "cuda" if torch.cuda.is_available() else "cpu"
embed_model = embed_model.to(device)

print(f"Embedding model loaded: {EMBEDDING_MODEL}")
print(f"Device: {device}")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Embedding model loaded: sentence-transformers/all-MiniLM-L6-v2
Device: cuda


In [31]:
def clean_prompt_for_retrieval(prompt):
    preamble_patterns = [
        r"^Pick the best possible answer:\s*",
        r"^Select the most accurate option:\s*",
        r"^Determine the correct option:\s*",
        r"^Identify the correct statement:\s*",
        r"^Which of the following is correct\?\s*",
        r"^Choose the correct answer:\s*"]
    for pattern in preamble_patterns:
        prompt = re.sub(pattern, "", prompt, flags=re.IGNORECASE)

    suffix_patterns = [
        r"\s*among the listed options\.?$",
        r"\s*from the following choices\.?$",
        r"\s*based on the given options\.?$",
        r"\s*carefully\.?$"]
    for pattern in suffix_patterns:
        prompt = re.sub(pattern, "", prompt, flags=re.IGNORECASE)

    return prompt.strip()

sample_prompt = test_df.iloc[0]["prompt"]
cleaned = clean_prompt_for_retrieval(sample_prompt)
print(f"Original: {sample_prompt[:100]}")
print(f"Cleaned:  {cleaned[:100]}")

Original: Pick the best possible answer: What is the relationship between the Hamiltonians and eigenstates in 
Cleaned:  What is the relationship between the Hamiltonians and eigenstates in supersymmetric quantum mechanic


In [32]:
def embed_queries(prompts, model, clean=True):
    if clean:
        prompts = [clean_prompt_for_retrieval(p) for p in prompts]
    
    embeddings = model.encode(prompts,batch_size=64,show_progress_bar=len(prompts) > 10,
                              normalize_embeddings=True, convert_to_numpy=True)
    return embeddings


print(f"Embedding {len(test_df)} test prompts...")
start_time = time.time()

test_prompts = test_df["prompt"].tolist()
test_embeddings = embed_queries(test_prompts, embed_model)

elapsed = time.time() - start_time
print(f"Done! Shape: {test_embeddings.shape} | Time: {elapsed:.2f}s")

print(f"\nEmbedding {len(train_df)} train prompts...")
train_prompts = train_df["prompt"].tolist()
train_embeddings = embed_queries(train_prompts, embed_model)
print(f"Done! Shape: {train_embeddings.shape}")

Embedding 500 test prompts...


Batches:   0%|          | 0/8 [00:00<?, ?it/s]

Done! Shape: (500, 384) | Time: 0.23s

Embedding 2000 train prompts...


Batches:   0%|          | 0/32 [00:00<?, ?it/s]

Done! Shape: (2000, 384)


---
## Step 6: FAISS Search — Retrieve Top-k=10 Passages

For each question, query the FAISS index to find the 10 most semantically similar Wikipedia chunks.

**Why k=10 and not k=3?**  
The bi-encoder (MiniLM) does fast but **approximate** semantic matching. It casts a wide net.  
In the next step, the cross-encoder will precision-filter these 10 candidates down to the best 3.

In [33]:
def retrieve_passages_lightweight(query_embeddings, index, top_k=10, batch_size=100):
    scores_list = []
    indices_list = []
    for start_idx in range(0, len(query_embeddings), batch_size):
        batch = query_embeddings[start_idx:start_idx + batch_size]
        scores, indices = index.search(batch, top_k)
        scores_list.append(scores)
        indices_list.append(indices)
    
    import numpy as np
    return np.vstack(scores_list), np.vstack(indices_list)


def get_passages_for_query(chunks, indices_row, scores_row):
    results = []
    for idx, score in zip(indices_row, scores_row):
        idx = int(idx)
        if idx < 0:
            continue
        results.append({
            "title":     chunks[idx]["title"],
            "text":      chunks[idx]["text"],
            "score":     float(score),
            "chunk_idx": idx})
    return results

print(f"Retrieval functions defined (with batching). Top-k = {FAISS_TOP_K}")

Retrieval functions defined (with batching). Top-k = 10


In [34]:
import gc
gc.collect()

print(f"Retrieving top-{FAISS_TOP_K} passages for {len(test_df)} test questions...")
import time
start_time = time.time()

test_scores, test_indices = retrieve_passages_lightweight(test_embeddings, index, top_k=FAISS_TOP_K)

elapsed = time.time() - start_time
print(f"Done! Shape: {test_indices.shape} | Time: {elapsed:.3f}s")

print(f"\nRetrieving top-{FAISS_TOP_K} passages for {len(train_df)} train questions...")

train_scores, train_indices = retrieve_passages_lightweight(train_embeddings, index, top_k=FAISS_TOP_K)

print(f"Done! Shape: {train_indices.shape}")

Retrieving top-10 passages for 500 test questions...
Done! Shape: (500, 10) | Time: 21.919s

Retrieving top-10 passages for 2000 train questions...
Done! Shape: (2000, 10)


In [35]:
sample_idx = 0
sample_prompt = test_df.iloc[sample_idx]["prompt"]

print(f"Sample question (test row {sample_idx}):")
print(f"  Prompt: {sample_prompt[:120]}...\n")
print(f"Top-{FAISS_TOP_K} retrieved passages:")

sample_passages = get_passages_for_query(
    all_chunks, test_indices[sample_idx], test_scores[sample_idx])

for rank, passage in enumerate(sample_passages, 1):
    print(f"  #{rank} | Score: {passage['score']:.4f} | Title: {passage['title']}")
    print(f" {passage['text'][:120]}...")

Sample question (test row 0):
  Prompt: Pick the best possible answer: What is the relationship between the Hamiltonians and eigenstates in supersymmetric quant...

Top-10 retrieved passages:
  #1 | Score: 0.6299 | Title: Supersymmetric quantum mechanics
 oscillator. a similar supersymmetric approach can also be used to more accurately find the hydrogen spectrum using the d...
  #2 | Score: 0.6098 | Title: Supersymmetric quantum mechanics
 we define the operators and where, which we need to choose, is called the superpotential of. we also define the aforemen...
  #3 | Score: 0.5727 | Title: Supersymmetric quantum mechanics
 , of course. then, incidentally, there ' s also a u ( 1 ) r symmetry, with p and x and w having zero r - charges and hav...
  #4 | Score: 0.5698 | Title: Supersymmetric quantum mechanics
 an introductory theorem shows that for every eigenstate of one hamiltonian, its partner hamiltonian has a corresponding ...
  #5 | Score: 0.5468 | Title: Supersymmetric quantum mec

In [36]:
test_unique_titles = []
for i in range(len(test_df)):
    titles = set(
        all_chunks[int(idx)]["title"] 
        for idx in test_indices[i] if idx >= 0
    )
    test_unique_titles.append(len(titles))

print(f"Retrieval diversity (unique article titles per query):")
print(f"  Mean: {np.mean(test_unique_titles):.1f} unique titles out of {FAISS_TOP_K}")
print(f"  Median: {np.median(test_unique_titles):.0f}")
print(f"  Min: {np.min(test_unique_titles)}")
print(f"  Max: {np.max(test_unique_titles)}")

print(f"\nRetrieval score statistics:")
print(f"  Mean: {np.mean(test_scores):.4f}")
print(f"  Median: {np.median(test_scores):.4f}")
print(f"  Min: {np.min(test_scores):.4f}")
print(f"  Max: {np.max(test_scores):.4f}")

Retrieval diversity (unique article titles per query):
  Mean: 5.3 unique titles out of 10
  Median: 5
  Min: 1
  Max: 10

Retrieval score statistics:
  Mean: 0.5826
  Median: 0.5791
  Min: 0.3659
  Max: 0.8578


---
## Step 7: Cross-Encoder Re-ranking → Top 3

The bi-encoder (MiniLM) retrieves candidates quickly but imprecisely.  
The **cross-encoder** processes each `(question, passage)` pair *jointly* through the full transformer,  
giving a much more accurate relevance score.

**Two-stage pipeline**:
1. Bi-encoder: 500K chunks → top 10 (fast, ~5ms)
2. Cross-encoder: 10 passages → top 3 (slower but precise, ~50ms per question)

**Model**: `cross-encoder/ms-marco-MiniLM-L-6-v2` (~22M params, runs on CPU)

In [37]:
cross_encoder = CrossEncoder(CROSS_ENCODER_MODEL)

print(f"Cross-encoder loaded: {CROSS_ENCODER_MODEL}")

config.json:   0%|          | 0.00/794 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/90.9M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: cross-encoder/ms-marco-MiniLM-L-6-v2
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/132 [00:00<?, ?B/s]

Cross-encoder loaded: cross-encoder/ms-marco-MiniLM-L-6-v2


In [38]:
def rerank_passages(query, passages, cross_encoder_model, top_n=3):
    if not passages:
        return []
    
    pairs = [[query, p["text"]] for p in passages]
    ce_scores = cross_encoder_model.predict(pairs)
    
    scored_passages = []
    for passage, score in zip(passages, ce_scores):
        scored_passage = passage.copy()
        scored_passage["ce_score"] = float(score)
        scored_passages.append(scored_passage)
    reranked = sorted(scored_passages, key=lambda x: x["ce_score"], reverse=True)
    return reranked[:top_n]


print(f"Reranking function defined. Top-n = {RERANK_TOP_N}")

Reranking function defined. Top-n = 2


In [39]:
print(f"Re-ranking passages for {len(test_df)} test questions...")
start_time = time.time()

test_reranked = []
for i in range(len(test_df)):
    prompt = test_df.iloc[i]["prompt"]
    
    passages = get_passages_for_query(all_chunks, test_indices[i], test_scores[i])
    
    reranked = rerank_passages(query=prompt,passages=passages,
                               cross_encoder_model=cross_encoder,top_n=RERANK_TOP_N)
    test_reranked.append(reranked)

    if (i + 1) % 100 == 0:
        elapsed = time.time() - start_time
        rate = (i + 1) / elapsed
        print(f"  [{i+1}/{len(test_df)}] {rate:.1f} questions/sec")

elapsed = time.time() - start_time
print(f"\nDone! Time: {elapsed:.1f}s")

Re-ranking passages for 500 test questions...
  [100/500] 40.0 questions/sec
  [200/500] 40.6 questions/sec
  [300/500] 40.8 questions/sec
  [400/500] 40.9 questions/sec
  [500/500] 40.9 questions/sec

Done! Time: 12.2s


In [40]:
print(f"Re-ranking passages for {len(train_df)} train questions...")
start_time = time.time()

train_reranked = []
for i in range(len(train_df)):
    prompt = train_df.iloc[i]["prompt"]
    passages = get_passages_for_query(all_chunks, train_indices[i], train_scores[i])
    reranked = rerank_passages(query=prompt,passages=passages,
                               cross_encoder_model=cross_encoder,top_n=RERANK_TOP_N)
    train_reranked.append(reranked)

    if (i + 1) % 500 == 0:
        elapsed = time.time() - start_time
        rate = (i + 1) / elapsed
        print(f"  [{i+1}/{len(train_df)}] {rate:.1f} questions/sec")

elapsed = time.time() - start_time
print(f"Done! Time: {elapsed:.1f}s")


Re-ranking passages for 2000 train questions...
  [500/2000] 40.5 questions/sec
  [1000/2000] 40.3 questions/sec
  [1500/2000] 40.2 questions/sec
  [2000/2000] 40.1 questions/sec
Done! Time: 49.9s


In [41]:
sample_idx = 0
sample_prompt = test_df.iloc[sample_idx]["prompt"]

print(f"Question (test row {sample_idx}):")
print(f"  {sample_prompt[:120]}...\n")

print(f"FAISS top-{FAISS_TOP_K} (bi-encoder scores):")
print("─" * 80)
sample_passages = get_passages_for_query(
    all_chunks, test_indices[sample_idx], test_scores[sample_idx]
)
for rank, p in enumerate(sample_passages, 1):
    print(f"  #{rank:2d} | Bi-enc: {p['score']:.4f} | {p['title'][:40]:40s} | {p['text'][:60]}...")

print(f"\nCross-encoder top-{RERANK_TOP_N} (reranked):")
print("─" * 80)
for rank, p in enumerate(test_reranked[sample_idx], 1):
    print(f"  #{rank} | CE: {p['ce_score']:.4f} | Bi-enc: {p['score']:.4f} | {p['title']}")
    print(f"      {p['text'][:100]}...")

Question (test row 0):
  Pick the best possible answer: What is the relationship between the Hamiltonians and eigenstates in supersymmetric quant...

FAISS top-10 (bi-encoder scores):
────────────────────────────────────────────────────────────────────────────────
  # 1 | Bi-enc: 0.6299 | Supersymmetric quantum mechanics         | oscillator. a similar supersymmetric approach can also be us...
  # 2 | Bi-enc: 0.6098 | Supersymmetric quantum mechanics         | we define the operators and where, which we need to choose, ...
  # 3 | Bi-enc: 0.5727 | Supersymmetric quantum mechanics         | , of course. then, incidentally, there ' s also a u ( 1 ) r ...
  # 4 | Bi-enc: 0.5698 | Supersymmetric quantum mechanics         | an introductory theorem shows that for every eigenstate of o...
  # 5 | Bi-enc: 0.5468 | Supersymmetric quantum mechanics         | of operators. we shall call this system supersymmetric if th...
  # 6 | Bi-enc: 0.5342 | Supersymmetric quantum mechanics         | of the 

In [42]:
rerank_changed_top1 = 0
for i in range(len(test_df)):
    faiss_top1_idx = int(test_indices[i][0]) if test_indices[i][0] >= 0 else None
    ce_top1_idx    = test_reranked[i][0]["chunk_idx"] if test_reranked[i] else None
    if faiss_top1_idx != ce_top1_idx:
        rerank_changed_top1 += 1

print(f"Reranking impact on test set:")
print(f"  Top-1 passage changed: {rerank_changed_top1}/{len(test_df)} "
      f"({rerank_changed_top1/len(test_df)*100:.1f}%)")
print(f"  Top-1 unchanged:       {len(test_df) - rerank_changed_top1}/{len(test_df)}")

Reranking impact on test set:
  Top-1 passage changed: 314/500 (62.8%)
  Top-1 unchanged:       186/500


---
## Step 8: Construct the Augmented Prompt

Build a structured prompt that combines:
1. **Retrieved context** (top 3 passages from the cross-encoder)
2. **Question** (the original prompt)
3. **Answer options** (A through E)

This prompt will be fed to the LLM in Step 9.

### Prompt Design Principles
- **Context before question** — LLMs attend better to recent tokens
- **Explicit instruction** — tell the model to use the context
- **Open-ended completion cue** — end with "The correct answer is" so we can extract logits for A/B/C/D/E
- **Token budget awareness** — keep total under 8K tokens (Mistral) or 512 (Flan-T5)

In [43]:
def build_augmented_prompt(question, options, passages, prompt_style="standard"):
    context_parts = []
    for i, p in enumerate(passages, 1):
        context_parts.append(f"[Source {i}: {p['title']}]\n{p['text']}")
    context_str = "\n\n".join(context_parts)
    
    if prompt_style == "flan":
        prompt = (
            f"Based on the following context, answer the multiple-choice question.\n\n"
            f"Context:\n{context_str}\n\n"
            f"Question: {question}\n\n"
            f"A: {options['A']}\n"
            f"B: {options['B']}\n"
            f"C: {options['C']}\n"
            f"D: {options['D']}\n"
            f"E: {options['E']}\n\n"
            f"The correct answer is"
        )
    else:
        prompt = (
            f"You are a knowledgeable assistant answering multiple-choice questions.\n"
            f"Use the provided context to select the most accurate answer.\n\n"
            f"### Context:\n{context_str}\n\n"
            f"### Question:\n{question}\n\n"
            f"### Options:\n"
            f"A: {options['A']}\n"
            f"B: {options['B']}\n"
            f"C: {options['C']}\n"
            f"D: {options['D']}\n"
            f"E: {options['E']}\n\n"
            f"### Answer:\n"
            f"The correct answer is"
        )
    
    return prompt


def build_prompt_for_row(df, row_idx, reranked_passages, prompt_style="standard"):
    row = df.iloc[row_idx]
    question = str(row["prompt"])
    options = {c: str(row[c]) for c in CHOICES}
    passages = reranked_passages[row_idx]
    return build_augmented_prompt(question, options, passages, prompt_style)


print("Prompt construction functions defined.")

Prompt construction functions defined.


In [44]:
sample_idx = 0

sample_prompt_standard = build_prompt_for_row(test_df, sample_idx, 
                                              test_reranked, prompt_style="standard")

sample_prompt_flan = build_prompt_for_row(test_df, sample_idx, 
                                          test_reranked, prompt_style="flan")

print("═" * 80)
print("STANDARD PROMPT (Mistral / LLaMA / Qwen)")
print("═" * 80)
print(sample_prompt_standard)
print(f"\n[Total characters: {len(sample_prompt_standard):,d}]")

print("\n")
print("═" * 80)
print("FLAN-T5 PROMPT")
print("═" * 80)
print(sample_prompt_flan)
print(f"\n[Total characters: {len(sample_prompt_flan):,d}]")

════════════════════════════════════════════════════════════════════════════════
STANDARD PROMPT (Mistral / LLaMA / Qwen)
════════════════════════════════════════════════════════════════════════════════
You are a knowledgeable assistant answering multiple-choice questions.
Use the provided context to select the most accurate answer.

### Context:
[Source 1: Supersymmetric quantum mechanics]
an introductory theorem shows that for every eigenstate of one hamiltonian, its partner hamiltonian has a corresponding eigenstate with the same energy ( except possibly for zero energy eigenstates ). this fact can be exploited to deduce many properties of the eigenstate spectrum. it is analogous to the original description of susy, which referred to bosons and fermions. we can imagine a " bosonic hamiltonian ", whose eigenstates are the various bosons of our theory. the susy partner of this hamiltonian would be " fermionic ", and its eigenstates would be the theory ' s fermions. each boson would ha

In [45]:
from transformers import AutoTokenizer

length_tokenizer = AutoTokenizer.from_pretrained(EMBEDDING_MODEL)

test_prompt_tokens = []
for i in range(len(test_df)):
    prompt = build_prompt_for_row(test_df, i, test_reranked, prompt_style="standard")
    tokens = length_tokenizer.encode(prompt, add_special_tokens=False)
    test_prompt_tokens.append(len(tokens))

test_prompt_tokens = np.array(test_prompt_tokens)

print("Augmented prompt token lengths (standard style):")
print(f"  Mean:   {test_prompt_tokens.mean():.0f} tokens")
print(f"  Median: {np.median(test_prompt_tokens):.0f} tokens")
print(f"  Min:    {test_prompt_tokens.min()} tokens")
print(f"  Max:    {test_prompt_tokens.max()} tokens")
print(f"  P95:    {np.percentile(test_prompt_tokens, 95):.0f} tokens")
print(f"  P99:    {np.percentile(test_prompt_tokens, 99):.0f} tokens")

print(f"\nContext window fit:")
for ctx_size in [512, 1024, 2048, 4096, 8192]:
    fit_count = (test_prompt_tokens <= ctx_size).sum()
    print(f"  ≤ {ctx_size:5d} tokens: {fit_count}/{len(test_df)} "
          f"({fit_count/len(test_df)*100:.1f}%)")

Token indices sequence length is longer than the specified maximum sequence length for this model (643 > 512). Running this sequence through the model will result in indexing errors


Augmented prompt token lengths (standard style):
  Mean:   652 tokens
  Median: 632 tokens
  Min:    363 tokens
  Max:    1080 tokens
  P95:    835 tokens
  P99:    952 tokens

Context window fit:
  ≤   512 tokens: 22/500 (4.4%)
  ≤  1024 tokens: 497/500 (99.4%)
  ≤  2048 tokens: 500/500 (100.0%)
  ≤  4096 tokens: 500/500 (100.0%)
  ≤  8192 tokens: 500/500 (100.0%)


In [46]:
PROMPT_STYLE = "flan"

print(f"Building augmented prompts (style={PROMPT_STYLE})...")

test_augmented_prompts = []
for i in range(len(test_df)):
    prompt = build_prompt_for_row(test_df, i, test_reranked, prompt_style=PROMPT_STYLE)
    test_augmented_prompts.append(prompt)

train_augmented_prompts = []
for i in range(len(train_df)):
    prompt = build_prompt_for_row(train_df, i, train_reranked, prompt_style=PROMPT_STYLE)
    train_augmented_prompts.append(prompt)

print(f"  Test augmented prompts:  {len(test_augmented_prompts)}")
print(f"  Train augmented prompts: {len(train_augmented_prompts)}")

Building augmented prompts (style=flan)...
  Test augmented prompts:  500
  Train augmented prompts: 2000


In [47]:
import json

test_prompts_path = os.path.join(ARTIFACTS_DIR, "test_augmented_prompts.pkl")
with open(test_prompts_path, "wb") as f:
    pickle.dump({
        "prompts":  test_augmented_prompts,
        "style":    PROMPT_STYLE,
        "reranked": test_reranked}, f)

train_prompts_path = os.path.join(ARTIFACTS_DIR, "train_augmented_prompts.pkl")
with open(train_prompts_path, "wb") as f:
    pickle.dump({
        "prompts":  train_augmented_prompts,
        "style":    PROMPT_STYLE,
        "reranked": train_reranked}, f)

test_size_mb  = os.path.getsize(test_prompts_path) / (1024 * 1024)
train_size_mb = os.path.getsize(train_prompts_path) / (1024 * 1024)

print(f"Saved test augmented prompts:  {test_prompts_path} ({test_size_mb:.1f} MB)")
print(f"Saved train augmented prompts: {train_prompts_path} ({train_size_mb:.1f} MB)")

Saved test augmented prompts:  /kaggle/working/test_augmented_prompts.pkl (2.0 MB)
Saved train augmented prompts: /kaggle/working/train_augmented_prompts.pkl (6.7 MB)


---
## Validation: Retrieval Hit Rate on Train Set

Since `train.csv` has ground truth answers, we can check if the retrieved passages
actually contain the correct answer text. This tells us how effective the RAG
retrieval is before the LLM even gets involved.

In [48]:
def check_retrieval_hit(row, reranked_passages, threshold=0.5):
    correct_answer = str(row[row["answer"]])
    answer_words = set(w.lower() for w in correct_answer.split() if len(w) > 3)
    
    if not answer_words:
        return False
    
    for passage in reranked_passages:
        passage_lower = passage["text"].lower()
        matched = sum(1 for w in answer_words if w in passage_lower)
        if matched / len(answer_words) >= threshold:
            return True
    
    return False

N_VAL = min(200, len(train_df))
hits = 0
for i in range(N_VAL):
    row = train_df.iloc[i]
    if check_retrieval_hit(row, train_reranked[i]):
        hits += 1

hit_rate = hits / N_VAL * 100
print(f"Retrieval hit rate (first {N_VAL} train rows):")
print(f"  Hits: {hits}/{N_VAL} ({hit_rate:.1f}%)")
print(f"\n  A hit rate of 50%+ is good for Wikipedia-based retrieval.")
print(f"  The LLM can still answer correctly without a perfect hit,")
print(f"  using its parametric knowledge combined with partial context.")

Retrieval hit rate (first 200 train rows):
  Hits: 89/200 (44.5%)

  A hit rate of 50%+ is good for Wikipedia-based retrieval.
  The LLM can still answer correctly without a perfect hit,
  using its parametric knowledge combined with partial context.


In [49]:
print("═" * 60)
print("  RAG PHASE 2 — RETRIEVAL & PROMPT CONSTRUCTION COMPLETE")
print("═" * 60)
print(f"")
print(f"  Step 4: Loaded FAISS index ({index.ntotal:,d} vectors)")
print(f"  Step 5: Embedded {len(test_df)} test + {len(train_df)} train prompts")
print(f"  Step 6: Retrieved top-{FAISS_TOP_K} passages per question")
print(f"  Step 7: Re-ranked to top-{RERANK_TOP_N} with cross-encoder")
print(f"           ({rerank_changed_top1}/{len(test_df)} test top-1s changed by reranking)")
print(f"  Step 8: Built augmented prompts (style={PROMPT_STYLE})")
print(f"           Median length: {np.median(test_prompt_tokens):.0f} tokens")
print(f"")
print(f"  Saved artifacts:")
print(f"    • {test_prompts_path}")
print(f"    • {train_prompts_path}")

════════════════════════════════════════════════════════════
  RAG PHASE 2 — RETRIEVAL & PROMPT CONSTRUCTION COMPLETE
════════════════════════════════════════════════════════════

  Step 4: Loaded FAISS index (646,914 vectors)
  Step 5: Embedded 500 test + 2000 train prompts
  Step 6: Retrieved top-10 passages per question
  Step 7: Re-ranked to top-2 with cross-encoder
           (314/500 test top-1s changed by reranking)
  Step 8: Built augmented prompts (style=flan)
           Median length: 632 tokens

  Saved artifacts:
    • /kaggle/working/test_augmented_prompts.pkl
    • /kaggle/working/train_augmented_prompts.pkl


## RAG_Step9_to_10_Inference

<!-- merged from: RAG_Step9_to_10_Inference.ipynb (Jupy Tools) -->

In [50]:
import pandas as pd
import numpy as np
import pickle
import time
import os
import torch
from transformers import AutoModelForCausalLM, AutoModelForSeq2SeqLM, AutoTokenizer

print("All imports successful!")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
else:
    print("Using CPU")

All imports successful!
GPU: Tesla T4


In [51]:
ARTIFACTS_DIR = "/kaggle/working" 
TRAIN_CSV = "/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv"
TEST_CSV = "/kaggle/input/competitions/smart-mcq-solver-challenge/test.csv"

MODEL_NAME = "google/flan-t5-large"
IS_SEQ2SEQ = "flan" in MODEL_NAME.lower() or "t5" in MODEL_NAME.lower()

CHOICES = ["A", "B", "C", "D", "E"]

print(f"Selected Model: {MODEL_NAME}")
print(f"Model Architecture: {'Encoder-Decoder (Seq2Seq)' if IS_SEQ2SEQ else 'Decoder-only (Causal)'}")

Selected Model: google/flan-t5-large
Model Architecture: Encoder-Decoder (Seq2Seq)


---
## Step 9: Load LLM & Augmented Prompts

In [52]:
test_prompts_path = os.path.join(ARTIFACTS_DIR, "test_augmented_prompts.pkl")
train_prompts_path = os.path.join(ARTIFACTS_DIR, "train_augmented_prompts.pkl")

with open(test_prompts_path, "rb") as f:
    test_data = pickle.load(f)
    test_augmented = test_data["prompts"]

with open(train_prompts_path, "rb") as f:
    train_data = pickle.load(f)
    train_augmented = train_data["prompts"]

train_df = pd.read_csv(TRAIN_CSV)
test_df  = pd.read_csv(TEST_CSV)

print(f"Loaded {len(test_augmented)} test prompts")
print(f"Loaded {len(train_augmented)} train prompts")

Loaded 500 test prompts
Loaded 2000 train prompts


In [53]:
print(f"Loading {MODEL_NAME}...")

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

if torch.cuda.is_available():
    device = "cuda"
    dtype = torch.float16
else:
    device = "cpu"
    dtype = torch.float32

if IS_SEQ2SEQ:
    model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_NAME, torch_dtype=dtype).to(device)
else:
    model = AutoModelForCausalLM.from_pretrained(
        MODEL_NAME, 
        torch_dtype=dtype, 
        device_map="auto" if device == "cuda" else None
    )
    if device != "cuda":
        model = model.to(device)

model.eval()
print(f"Model loaded on {device}")

Loading google/flan-t5-large...


config.json:   0%|          | 0.00/662 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

...

model.safetensors:   0%|          | 0.00/3.13G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/558 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

Model loaded on cuda


In [54]:
def get_letter_token_id(letter, tokenizer):
    tokens = tokenizer.encode(letter, add_special_tokens=False)
    if len(tokens) == 1:
        return tokens[0]
    
    for t in tokens:
        if tokenizer.decode([t]).strip() == letter:
            return t

    return tokens[0]

vocab_ids = {letter: get_letter_token_id(letter, tokenizer) for letter in CHOICES}

print("Token IDs for answer choices:")
for k, v in vocab_ids.items():
    print(f"{k}: {v} -> '{tokenizer.decode([v])}'")

Token IDs for answer choices:
A: 71 -> 'A'
B: 272 -> 'B'
C: 205 -> 'C'
D: 309 -> 'D'
E: 262 -> 'E'


---
## Step 10: Extract Logits & Calculate MAP@3

We feed the prompt into the model, but instead of using `.generate()` to predict arbitrary text, we look at the raw probability distribution (logits) for the very next token, specifically for the tokens corresponding to A, B, C, D, and E.

In [55]:
def get_answer_probs(prompt, tokenizer, model, vocab_ids, is_seq2seq):
    tokenizer.truncation_side = "left"
    inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=2048).to(model.device)
    
    with torch.no_grad():
        if is_seq2seq:
            decoder_input_ids = tokenizer("", return_tensors="pt").input_ids[:, :1].to(model.device)
            outputs = model(**inputs, decoder_input_ids=decoder_input_ids)
            last_logits = outputs.logits[0, -1, :]
        else:
            outputs = model(**inputs)
            last_logits = outputs.logits[0, -1, :]
            
    answer_logits = torch.tensor([last_logits[vocab_ids[letter]].item() for letter in CHOICES])
    probs = torch.softmax(answer_logits, dim=0)
    
    prob_dict = {letter: p.item() for letter, p in zip(CHOICES, probs)}
    ranked = sorted(prob_dict.items(), key=lambda x: x[1], reverse=True)
    
    return ranked

sample_ranked = get_answer_probs(train_augmented[0], tokenizer, model, vocab_ids, IS_SEQ2SEQ)
print("Sample prediction probabilities:")
for letter, prob in sample_ranked:
    print(f"  {letter}: {prob:.4f}")

Sample prediction probabilities:
  A: 0.2169
  E: 0.2045
  C: 0.2013
  D: 0.2007
  B: 0.1766


In [56]:
def map_at_3(true_answer, predicted_top3):
    score = 0.0
    for i, pred in enumerate(predicted_top3):
        if pred == true_answer:
            score += 1.0 / (i + 1)
            break
    return score

print("MAP@3 scoring function defined.")

MAP@3 scoring function defined.


In [57]:
N_VAL = min(100, len(train_df))

print(f"Running inference on {N_VAL} validation samples...")
start_time = time.time()

val_map3_scores = []
val_predictions = []

for i in range(N_VAL):
    prompt = train_augmented[i]
    true_ans = train_df.iloc[i]["answer"]
    
    ranked = get_answer_probs(prompt, tokenizer, model, vocab_ids, IS_SEQ2SEQ)
    top3 = [r[0] for r in ranked[:3]]
    
    val_predictions.append(top3)
    score = map_at_3(true_ans, top3)
    val_map3_scores.append(score)
    
    if (i + 1) % 10 == 0:
        elapsed = time.time() - start_time
        print(f"  [{i+1}/{N_VAL}] MAP@3: {np.mean(val_map3_scores):.4f} | Time: {elapsed:.1f}s")

print("\n" + "═"*40)
print(f"Final Validation MAP@3 ({N_VAL} samples): {np.mean(val_map3_scores):.4f}")
print("═"*40)

Running inference on 100 validation samples...
  [10/100] MAP@3: 0.4333 | Time: 1.6s
  [20/100] MAP@3: 0.3167 | Time: 3.2s
  [30/100] MAP@3: 0.3167 | Time: 5.0s
  [40/100] MAP@3: 0.3375 | Time: 6.6s
  [50/100] MAP@3: 0.3067 | Time: 8.4s
  [60/100] MAP@3: 0.3167 | Time: 10.2s
  [70/100] MAP@3: 0.3167 | Time: 11.7s
  [80/100] MAP@3: 0.2938 | Time: 13.3s
  [90/100] MAP@3: 0.3074 | Time: 14.9s
  [100/100] MAP@3: 0.3283 | Time: 16.6s

════════════════════════════════════════
Final Validation MAP@3 (100 samples): 0.3283
════════════════════════════════════════


In [58]:
print(f"Running inference on all {len(test_df)} test samples...")
start_time = time.time()

test_predictions = []

for i in range(len(test_df)):
    prompt = test_augmented[i]
    ranked = get_answer_probs(prompt, tokenizer, model, vocab_ids, IS_SEQ2SEQ)
    top3 = [r[0] for r in ranked[:3]]
    test_predictions.append(" ".join(top3))
    
    if (i + 1) % 50 == 0:
        print(f"  [{i+1}/{len(test_df)}] Processed")

elapsed = time.time() - start_time
print(f"\nDone! Total time: {elapsed:.1f}s")

Running inference on all 500 test samples...
  [50/500] Processed
  [100/500] Processed
  [150/500] Processed
  [200/500] Processed
  [250/500] Processed
  [300/500] Processed
  [350/500] Processed
  [400/500] Processed
  [450/500] Processed
  [500/500] Processed

Done! Total time: 80.4s


In [59]:
submission_df = pd.DataFrame({
    "id": test_df["id"] if "id" in test_df.columns else range(len(test_df)),
    "prediction": test_predictions})

submission_path = os.path.join(ARTIFACTS_DIR, "submission.csv")
submission_df.to_csv(submission_path, index=False)

print(f"Saved submission to {submission_path}")
submission_df.head()

Saved submission to /kaggle/working/submission.csv


,id,prediction
0,1,D A B
1,2,A E D
2,3,D A C
3,4,A D B
4,5,A D B
